In [7]:
!pip install pandas numpy tensorflow scikit-learn matplotlib seaborn

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [26]:
file_path = "downloads/Grocery_Dataset.csv"
df = pd.read_csv(file_path)
df.head()

,Product_ID,Product_Name,Catagory,Supplier_ID,Supplier_Name,Stock,Reorder_Level,Reorder_Quantity,Price,Date_Received
0,29-205-1132,Sushi Rice,Grains & Pulses,38-037-1699,Jaxnation,22.0,72.0,70.0,NaN,8/16/2024
1,40-681-9981,Arabica Coffee,Beverages,54-470-2479,Feedmix,45.0,77.0,2.0,NaN,11/1/2024
2,06-955-3428,Black Rice,Grains & Pulses,54-031-2945,Vinder,30.0,38.0,83.0,NaN,8/3/2024
3,71-594-6552,Long Grain Rice,Grains & Pulses,63-492-7603,Brightbean,12.0,59.0,62.0,NaN,12/8/2024
4,57-437-1828,Plum,Fruits & Vegetables,54-226-4308,Topicstorm,37.0,30.0,74.0,NaN,7/3/2024


In [27]:
# Missing values hatayein aur Needs_Reorder column banayein
df = df.drop(columns=['Price'], errors='ignore').dropna(subset=['Stock', 'Reorder_Level', 'Reorder_Quantity', 'Catagory'])
df['Needs_Reorder'] = (df['Stock'] <= df['Reorder_Level']).astype(int)

df[['Catagory', 'Stock', 'Reorder_Level', 'Needs_Reorder']].head()

,Catagory,Stock,Reorder_Level,Needs_Reorder
0,Grains & Pulses,22.0,72.0,1
1,Beverages,45.0,77.0,1
2,Grains & Pulses,30.0,38.0,1
3,Grains & Pulses,12.0,59.0,1
4,Fruits & Vegetables,37.0,30.0,0


In [28]:
X = df[['Catagory', 'Stock', 'Reorder_Level', 'Reorder_Quantity']]
y = df['Needs_Reorder']

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Training rows:", len(X_train), "| Testing rows:", len(X_test))


Training rows: 786 | Testing rows: 197


In [29]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Stock', 'Reorder_Level', 'Reorder_Quantity']),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Catagory'])
    ]
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
print("Processed Features Shape:", X_train_proc.shape)

Processed Features Shape: (786, 10)


In [30]:
model = Sequential([
    Input(shape=(X_train_proc.shape[1],)),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,329 (13.00 KB)

 Trainable params: 3,329 (13.00 KB)

 Non-trainable params: 0 (0.00 B)

In [31]:
history = model.fit(
    X_train_proc, y_train,
    epochs=30,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

Epoch 1/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.7659 - loss: 0.6247 - val_accuracy: 0.8861 - val_loss: 0.5208
Epoch 2/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8981 - loss: 0.4071 - val_accuracy: 0.9430 - val_loss: 0.2672
Epoch 3/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9331 - loss: 0.1978 - val_accuracy: 0.9873 - val_loss: 0.1387
Epoch 4/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9506 - loss: 0.1389 - val_accuracy: 0.9873 - val_loss: 0.1007
Epoch 5/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9713 - loss: 0.1020 - val_accuracy: 0.9873 - val_loss: 0.0832
Epoch 6/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9713 - loss: 0.0823 - val_accuracy: 0.9810 - val_loss: 0.0763
Epoch 7/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9729 - loss: 0.0765 - val_accuracy: 0.9810 - val_loss: 0.0641
Epoch 8/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9729 - loss: 0.0648 - val_accuracy: 0.9810 - val_loss

In [32]:
loss, accuracy = model.evaluate(X_test_proc, y_test, verbose=0)
print(f"Final Model Accuracy: {accuracy * 100:.2f}%")

# Naye item par prediction testing
def predict_stock(category, stock, reorder_level, reorder_qty):
    sample = pd.DataFrame([{'Catagory': category, 'Stock': stock, 'Reorder_Level': reorder_level, 'Reorder_Quantity': reorder_qty}])
    sample_proc = preprocessor.transform(sample)
    prob = model.predict(sample_proc, verbose=0)[0][0]
    return "REORDER NEEDED" if prob > 0.5 else "STOCK SUFFICIENT", prob

status, score = predict_stock('Beverages', stock=10.0, reorder_level=50.0, reorder_qty=40.0)
print(f"Prediction Result: {status} (Confidence: {score:.2f})")

Final Model Accuracy: 99.49%
Prediction Result: REORDER NEEDED (Confidence: 1.00)


In [33]:
import os
print([f for f in os.listdir() if f.endswith(('.keras', '.joblib'))])
print([f for f in os.listdir('downloads') if f.endswith(('.keras', '.joblib'))])

['.keras']
[]


In [36]:
import joblib

model.save("grocery_ann_model.keras")
model.save("downloads/grocery_ann_model.keras")

joblib.dump(preprocessor, "preprocessor.joblib")
joblib.dump(preprocessor, "downloads/preprocessor.joblib")

print("Files saved successfully in all paths!")

Files saved successfully in all paths!


In [35]:
import os
print([f for f in os.listdir() if f.endswith(('.keras', '.joblib'))])

['preprocessor.joblib', 'grocery_ann_model.keras', '.keras']
